# Entity salience checker: is your page about the entity you think it is?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lazarinastoy/mlforseo-code/blob/main/entity-salience-checker/entity_salience_checker.ipynb)

A free notebook by **Lazarina Stoy, MLforSEO**. It's a lite version of one check from the **Clarity** step of the 5C framework (Clarity, Coverage, Consistency, Connection, Corroboration) in my course [The AI-powered Entity Audit](https://www.mlforseo.com/courses/the-ai-powered-entity-audit/).

**The question:** when a machine reads your page, is the entity you wrote it about the one that comes out on top? Or does your brand name, your newsletter box or a synonym you used half the time win instead?

**What it does, for each URL:**

1. Fetches the page and extracts the **main text** with `trafilatura` (optionally the full page text too, navigation and footer included, so you can compare).
2. Sends the text to the **Google Cloud Natural Language API v1** `documents:analyzeEntities`.
3. Ranks the entities by **salience**, finds your **target entity** (by name, aliases, Wikipedia URL or Knowledge Graph MID) and reports where it ranks.
4. Compares your page with **competitor pages** for the same topic.
5. Flags boilerplate entities (brand, newsletter, shipping, cookies...) that outrank the target, and gives simple **clarity tips**: mention the target early, in the title and headings, and with one unambiguous name.
6. Exports a Sheets-ready `.xlsx` and CSVs.

**Why v1?** Salience is only returned by the v1 API. The v2 `Entity` object has `name`, `type`, `metadata`, `mentions` and `sentiment`, and no `salience` field (checked against the official v1 and v2 REST references and the v1/v2 discovery documents, October 2026). This notebook calls v1 on purpose.

**Be clear about what salience is:** it's the Natural Language API's own estimate (0 to 1) of how central an entity is to the text you send. It's a useful, repeatable proxy for "what is this page about to a machine". Google has never said that Search or its AI features use this score, and LLM-based search systems certainly don't call this API. Treat it as a diagnostic, not a ranking factor.

**Sample mode** (`USE_SAMPLE = True`) runs without a key: three **fictional** pages about the French press, with **sample** API responses saved in `sample_data/responses/`. They follow the v1 response schema exactly, but the salience numbers come from a stand-in formula, not from Google (see `sample_data/make_sample_responses.py`). The Knowledge Graph MIDs and Wikipedia URLs in them are real.

In [1]:
# Installs trafilatura if it's missing (Colab doesn't ship it). Locally: pip install -r requirements.txt
import importlib.util
if importlib.util.find_spec('trafilatura') is None or importlib.util.find_spec('lxml_html_clean') is None:
    %pip install -q trafilatura lxml_html_clean

In [2]:
import os, re, json, math, hashlib, unicodedata, getpass, pathlib
import requests
import pandas as pd
import trafilatura
from bs4 import BeautifulSoup

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 160)

## 1. Settings

List the pages to check. For each one give:

* `url` and `role` (`yours` or `competitor`).
* `target_entity`: the entity the page should be about, in its most common, unambiguous name.
* `target_aliases`: other names you use for it, separated by `;` (synonyms, regional names, abbreviations).
* `target_wikipedia_url` (optional): the entity's Wikipedia page. When the API links an entity to the same page, it counts as a match even if the name differs. You can use `target_mid` instead if you know the Knowledge Graph ID.

Pages for the same topic should share the same target, so the comparison is fair. You can also load the list from a CSV with these columns (`PAGES_CSV`).

In [3]:
USE_SAMPLE = True                 # True: fictional sample pages + saved sample responses, no key needed
ANALYSE_TEXT = 'main'             # 'main' = article text (trafilatura); 'full' = every visible text on the page
COMPARE_FULL_TEXT = True          # also analyse the full page text for your pages (doubles API units for them)
TOP_N = 10                        # entities to show per page
LANGUAGE = ''                     # e.g. 'en'; empty lets the API detect it

PAGES = [
    # url, role, target_entity, target_aliases, target_wikipedia_url
    {'url': 'https://www.example.com/your-page/', 'role': 'yours', 'target_entity': 'French press',
     'target_aliases': 'cafetière; press pot', 'target_wikipedia_url': 'https://en.wikipedia.org/wiki/French_press'},
    {'url': 'https://www.competitor.example/their-page/', 'role': 'competitor', 'target_entity': 'French press',
     'target_aliases': 'cafetière; press pot', 'target_wikipedia_url': 'https://en.wikipedia.org/wiki/French_press'},
]
PAGES_CSV = ''                    # optional: path to a CSV with the same columns

SAMPLE_DIR = 'sample_data'
SAMPLE_RAW = 'https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/entity-salience-checker/sample_data'
CACHE_DIR = 'nl_api_responses'    # live responses are saved here so re-runs don't cost units again

## 2. API key (live mode only)

1. In Google Cloud, create or pick a project, enable the **Cloud Natural Language API** and make sure billing is on.
2. Create an **API key** (APIs & Services > Credentials). Restrict it to the Cloud Natural Language API.
3. In Colab, add it as a secret named `GOOGLE_NL_API_KEY` (key icon in the left sidebar) and allow this notebook to read it. Otherwise the cell asks for it. The key is never printed or saved.

**Cost:** the API bills in units of up to 1,000 characters per request, rounded up (a 4,200-character article is 5 units). Entity analysis includes **5,000 free units a month**; above that, check the current rate on the [official pricing page](https://cloud.google.com/natural-language/pricing). The notebook estimates units before it calls the API, and caches every response so re-running costs nothing.

In [4]:
API_KEY = None
if not USE_SAMPLE:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GOOGLE_NL_API_KEY')
    except Exception:
        API_KEY = os.environ.get('GOOGLE_NL_API_KEY')
    if not API_KEY:
        API_KEY = getpass.getpass('Google Cloud Natural Language API key: ')
    print('API key loaded.')
else:
    print('Sample mode: no API key needed.')

Sample mode: no API key needed.


## 3. Fetch the pages and extract the text

* **Main text** uses `trafilatura`, which keeps the article and drops navigation, sidebars and footers. Anything placed *inside* the article (newsletter boxes, product CTAs, author bios) usually stays, and that's often where boilerplate entities come from.
* **Full text** keeps every visible text node, so you can see how much the template itself changes what the page looks like it's about.
* **Boilerplate text** (nav, header, footer, aside and forms) is kept separately so the notebook can flag entities that live there.
* The **title, H1 and H2/H3 headings** are read from the HTML for the clarity checks.

In [5]:
def load_pages():
    if USE_SAMPLE:
        path = os.path.join(SAMPLE_DIR, 'pages_SAMPLE.csv')
        return pd.read_csv(path if os.path.exists(path) else f'{SAMPLE_RAW}/pages_SAMPLE.csv').fillna('')
    if PAGES_CSV:
        return pd.read_csv(PAGES_CSV).fillna('')
    return pd.DataFrame(PAGES).fillna('')


def get_html(row):
    if USE_SAMPLE:
        local = os.path.join(SAMPLE_DIR, 'html', f"{row['sample_id']}.html")
        if os.path.exists(local):
            return pathlib.Path(local).read_text(encoding='utf-8')
        return requests.get(f"{SAMPLE_RAW}/html/{row['sample_id']}.html", timeout=30).text
    resp = requests.get(row['url'], timeout=30, headers={'User-Agent': 'Mozilla/5.0 (compatible; MLforSEO-entity-salience-checker)'})
    resp.raise_for_status()
    return resp.text


def extract_main_text(html):
    return trafilatura.extract(html, include_comments=False, include_tables=False, favor_precision=True) or ''


def extract_full_text(html):
    soup = BeautifulSoup(html, 'html.parser')
    for tag in soup(['script', 'style', 'noscript', 'template']):
        tag.decompose()
    lines = [ln.strip() for ln in soup.get_text('\n').splitlines()]
    return '\n'.join(ln for ln in lines if ln)


def page_structure(html):
    soup = BeautifulSoup(html, 'html.parser')
    boiler = ' '.join(el.get_text(' ', strip=True) for el in soup.find_all(['nav', 'header', 'footer', 'aside', 'form']))
    return {
        'title': soup.title.get_text(strip=True) if soup.title else '',
        'h1': ' | '.join(h.get_text(' ', strip=True) for h in soup.find_all('h1')),
        'subheadings': ' | '.join(h.get_text(' ', strip=True) for h in soup.find_all(['h2', 'h3'])),
        'boilerplate_text': boiler,
    }


pages = load_pages()
records = []
for _, row in pages.iterrows():
    html = get_html(row)
    rec = dict(row)
    rec.update(page_structure(html))
    rec['main_text'] = extract_main_text(html)
    rec['full_text'] = extract_full_text(html)
    records.append(rec)
    print(f"{row['role']:<10} {row['url']}  main: {len(rec['main_text']):,} chars, full: {len(rec['full_text']):,} chars")
pages = pd.DataFrame(records)
if USE_SAMPLE:
    print('\nSAMPLE pages: fictional sites written for this notebook.')

yours      https://www.brewly.example/blog/how-to-use-a-cafetiere/  main: 1,883 chars, full: 2,289 chars
competitor https://www.coffee-one.example/guides/how-to-use-a-french-press/  main: 1,542 chars, full: 1,646 chars
competitor https://www.dailygrindnotes.example/brew/french-press-coffee/  main: 940 chars, full: 1,104 chars

SAMPLE pages: fictional sites written for this notebook.


## 4. Estimate API units, then analyse entities

Each request is billed in units of up to 1,000 characters (rounded up per request). The cell prints the estimate first. In live mode, responses are cached in `nl_api_responses/` by a hash of the text, so running the notebook again on unchanged pages is free.

The request uses `encodingType: UTF32`, so every mention's `beginOffset` is a character position in Python terms. That's what the "mentioned early?" check uses.

In [6]:
NL_URL = 'https://language.googleapis.com/v1/documents:analyzeEntities'   # v1: returns salience

def units(text):
    return max(1, math.ceil(len(text) / 1000))

jobs = [(i, ANALYSE_TEXT) for i in pages.index]
if COMPARE_FULL_TEXT and ANALYSE_TEXT == 'main':
    jobs += [(i, 'full') for i in pages.index[pages['role'] == 'yours']]
total_units = sum(units(pages.at[i, f'{mode}_text']) for i, mode in jobs)
print(f'{len(jobs)} requests, about {total_units} units (5,000 units a month are free for entity analysis).')


def analyze_entities(text):
    """Call the Cloud Natural Language API v1 analyzeEntities, with a local cache."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    key = hashlib.sha256((LANGUAGE + '|' + text).encode('utf-8')).hexdigest()[:20]
    cache = os.path.join(CACHE_DIR, f'{key}.json')
    if os.path.exists(cache):
        return json.load(open(cache, encoding='utf-8'))
    document = {'type': 'PLAIN_TEXT', 'content': text}
    if LANGUAGE:
        document['language'] = LANGUAGE
    resp = requests.post(NL_URL, json={'document': document, 'encodingType': 'UTF32'},
                         headers={'X-Goog-Api-Key': API_KEY}, timeout=60)   # key in a header, not the URL
    if resp.status_code != 200:
        raise RuntimeError(f'API error {resp.status_code}: {resp.text[:300]}')
    data = resp.json()
    json.dump(data, open(cache, 'w', encoding='utf-8'), ensure_ascii=False)
    return data


def load_sample_response(sample_id, mode):
    local = os.path.join(SAMPLE_DIR, 'responses', f'{sample_id}__{mode}.json')
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))
    return requests.get(f'{SAMPLE_RAW}/responses/{sample_id}__{mode}.json', timeout=30).json()


responses = {}
for i, mode in jobs:
    if USE_SAMPLE:
        responses[(i, mode)] = load_sample_response(pages.at[i, 'sample_id'], mode)
    else:
        responses[(i, mode)] = analyze_entities(pages.at[i, f'{mode}_text'])
print('Done.', 'Using SAMPLE responses (v1 schema, stand-in salience values).' if USE_SAMPLE else '')

4 requests, about 8 units (5,000 units a month are free for entity analysis).
Done. Using SAMPLE responses (v1 schema, stand-in salience values).


## 5. Flatten the entities and find the target

One row per entity per page, ranked by salience. For each entity:

* `salience`, `type`, number of `mentions` and how many are `PROPER` (a name) vs `COMMON` (a common noun).
* `mid` and `wikipedia_url` when the API linked the entity to Google's Knowledge Graph. Linked entities are ones the API could disambiguate.
* `first_mention_pct`: how far into the text the first mention appears (0% = the very start).
* `is_target`: matches your target by name, alias, Wikipedia URL or MID (`is_primary` marks the row for the main name or the Knowledge Graph match). If several entity rows match (for example "French press" and "cafetière" detected separately), the page's attention on your topic is **split** across them.
* `boilerplate_hint`: a typical template term (newsletter, shipping, cookies...), or a brand or proper name that also appears in the page's navigation, header, footer or sidebar. These are the entities that, as Beatrice Gamba showed, can end up outranking your actual topic.

In [7]:
BOILERPLATE_TERMS = {'newsletter', 'cookie', 'cookies', 'privacy policy', 'privacy', 'shipping', 'free shipping', 'basket',
                     'cart', 'account', 'login', 'subscribe', 'subscribers', 'discount', 'returns', 'terms'}

def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+', ' ', s).strip()

def target_names(row):
    names = [row['target_entity']] + [a for a in str(row.get('target_aliases', '')).split(';')]
    return {norm(n) for n in names if norm(n)}

def is_primary_entity(ent, row):
    # The entity row for the target's main name or its Knowledge Graph match (not an alias)
    meta = ent.get('metadata', {}) or {}
    wiki = str(row.get('target_wikipedia_url', '')).strip()
    mid = str(row.get('target_mid', '')).strip()
    return ((wiki and meta.get('wikipedia_url', '').rstrip('/') == wiki.rstrip('/'))
            or (mid and meta.get('mid') == mid) or norm(ent['name']) == norm(row['target_entity']))

def is_target_entity(ent, row):
    meta = ent.get('metadata', {}) or {}
    wiki = str(row.get('target_wikipedia_url', '')).strip()
    mid = str(row.get('target_mid', '')).strip()
    if wiki and meta.get('wikipedia_url', '').rstrip('/') == wiki.rstrip('/'):
        return True
    if mid and meta.get('mid') == mid:
        return True
    return norm(ent['name']) in target_names(row)

def flatten(i, mode):
    row = pages.loc[i]
    text = row[f'{mode}_text']
    boiler = norm(row['boilerplate_text'])
    out = []
    ranked = [e for e in responses[(i, mode)].get('entities', []) if e.get('type') not in ('NUMBER', 'PRICE', 'DATE', 'PHONE_NUMBER', 'ADDRESS')]
    ranked.sort(key=lambda e: e.get('salience', 0), reverse=True)
    for rank, e in enumerate(ranked, start=1):
        mentions = e.get('mentions', [])
        offsets = [m['text'].get('beginOffset', 0) for m in mentions if 'text' in m]
        name_n = norm(e['name'])
        out.append({
            'url': row['url'], 'role': row['role'], 'text_mode': mode, 'rank': rank,
            'entity': e['name'], 'type': e.get('type', ''), 'salience': round(e.get('salience', 0), 4),
            'mentions': len(mentions), 'proper_mentions': sum(m.get('type') == 'PROPER' for m in mentions),
            'mention_texts': ' | '.join(sorted({m['text']['content'] for m in mentions if 'text' in m})),
            'first_mention_pct': round(100 * min(offsets) / max(len(text), 1), 1) if offsets else None,
            'mid': (e.get('metadata') or {}).get('mid', ''),
            'wikipedia_url': (e.get('metadata') or {}).get('wikipedia_url', ''),
            'is_target': is_target_entity(e, row),
            'is_primary': bool(is_primary_entity(e, row)),
            # template term, or a brand/proper name that also appears in nav, header, footer or sidebar
            'boilerplate_hint': bool(name_n) and (name_n in BOILERPLATE_TERMS or (
                f' {name_n} ' in f' {boiler} ' and (e.get('type') == 'ORGANIZATION' or any(m.get('type') == 'PROPER' for m in mentions)))),
        })
    return out

entities = pd.DataFrame([r for (i, mode) in responses for r in flatten(i, mode)])
main_entities = entities[entities['text_mode'] == ANALYSE_TEXT]
for url, g in main_entities.groupby('url', sort=False):
    print(f'\n{url}')
    display(g.head(TOP_N)[['rank', 'entity', 'type', 'salience', 'mentions', 'mid', 'is_target', 'boilerplate_hint']])


https://www.brewly.example/blog/how-to-use-a-cafetiere/


,rank,entity,type,salience,mentions,mid,is_target,boilerplate_hint
0,1,Brewly,ORGANIZATION,0.1954,7,,False,True
1,2,cafetière,CONSUMER_GOOD,0.0743,5,,True,False
2,3,coffee,CONSUMER_GOOD,0.0731,5,/m/02vqfm,False,False
3,4,coffee beans,CONSUMER_GOOD,0.0549,3,/m/05bzrs,False,False
4,5,water,OTHER,0.0510,3,/m/0838f,False,False
5,6,grind,OTHER,0.0495,5,,False,False
6,7,filter,OTHER,0.0480,5,,False,False
7,8,Brewly Classic,CONSUMER_GOOD,0.0464,2,,False,False
8,9,brewers,CONSUMER_GOOD,0.0420,2,,False,False
9,10,burr grinder,CONSUMER_GOOD,0.0347,2,,False,False



https://www.coffee-one.example/guides/how-to-use-a-french-press/


,rank,entity,type,salience,mentions,mid,is_target,boilerplate_hint
29,1,French press,CONSUMER_GOOD,0.2993,15,/m/01p1qs,True,False
30,2,coffee,CONSUMER_GOOD,0.1645,9,/m/02vqfm,False,False
31,3,water,OTHER,0.0785,4,/m/0838f,False,False
32,4,filter,OTHER,0.0478,4,,False,False
33,5,grind,OTHER,0.0462,4,,False,False
34,6,plunger,OTHER,0.0423,4,,False,False
35,7,Bodum,ORGANIZATION,0.0347,1,/m/0cb7xp,False,False
36,8,brew time,OTHER,0.0248,2,,False,False
37,9,ratio,OTHER,0.0246,2,,False,False
38,10,coffee beans,CONSUMER_GOOD,0.0228,1,/m/05bzrs,False,False



https://www.dailygrindnotes.example/brew/french-press-coffee/


,rank,entity,type,salience,mentions,mid,is_target,boilerplate_hint
51,1,coffee,CONSUMER_GOOD,0.2376,6,/m/02vqfm,False,False
52,2,French press,CONSUMER_GOOD,0.1769,5,/m/01p1qs,True,False
53,3,brew time,OTHER,0.0987,5,,False,False
54,4,water,OTHER,0.0945,3,/m/0838f,False,False
55,5,grind,OTHER,0.0904,4,,False,False
56,6,ratio,OTHER,0.0746,3,,False,False
57,7,filter,OTHER,0.0560,3,,False,False
58,8,cup,OTHER,0.0388,2,,False,False
59,9,blade grinders,CONSUMER_GOOD,0.0323,1,,False,False
60,10,burr grinder,CONSUMER_GOOD,0.0300,1,,False,False


## 6. Clarity check per page

For each page:

* `top_entity` and its salience.
* `target_rank`: the rank of the target's main entity row (its main name, or the row linked to the same Wikipedia page / MID). `best_alias_rank` is the best rank of any matching row, aliases included.
* `target_salience`: salience summed over all matching rows. `primary_share` is how much of that sits on the main row.
* `target_split_into`: how many separate entity rows your target was detected as. More than 1, with a low `primary_share`, means the API didn't connect your synonyms to one entity, so the page's attention on its topic is spread thin.
* `kg_linked`: the API linked the target to a Knowledge Graph entity (it returned a MID or Wikipedia URL).
* `first_mention_pct`, and whether the target (or an alias) appears in the **title**, **H1** and **subheadings**.
* `outranked_by_boilerplate`: template-type entities with higher salience than the target.

**Lite clarity score (0-3)**, this notebook's own simple rubric:

| Score | Meaning |
|---|---|
| 3 | Target is the most salient entity, linked to the Knowledge Graph, and first mentioned in the first 10% of the text |
| 2 | Target is in the top 3 (or #1 but missing one of the other conditions) |
| 1 | Target is detected but outside the top 3 |
| 0 | Target isn't detected at all |

The full Clarity scoring in the course looks at more than this (placement rules, mention counts, intended vs detected entity across a whole site), so read this as a quick first pass.

In [8]:
def contains_any(text, names):
    t = f' {norm(text)} '
    return any(f' {n} ' in t for n in names)

def clarity_row(i, mode):
    row = pages.loc[i]
    ents = entities[(entities['url'] == row['url']) & (entities['text_mode'] == mode)]
    tgt = ents[ents['is_target']]
    names = target_names(row)
    top = ents.iloc[0] if len(ents) else None
    res = {'url': row['url'], 'role': row['role'], 'text_mode': mode, 'target_entity': row['target_entity'],
           'top_entity': top['entity'] if top is not None else '', 'top_salience': top['salience'] if top is not None else 0}
    if tgt.empty:
        res.update(target_rank=None, best_alias_rank=None, target_salience=0.0, primary_share=0.0, target_split_into=0,
                   kg_linked=False, first_mention_pct=None, names_used='', outranked_by_boilerplate='')
    else:
        primary = tgt[tgt['is_primary']]
        best = primary.iloc[0] if len(primary) else tgt.iloc[0]   # rank of the main name / KG match
        above = ents[ents['rank'] < best['rank']]
        total = tgt['salience'].sum()
        res.update(
            target_rank=int(best['rank']),
            best_alias_rank=int(tgt['rank'].min()),
            target_salience=round(total, 4),
            primary_share=round(primary['salience'].sum() / total, 2) if total else 0.0,
            target_split_into=len(tgt),
            kg_linked=bool((tgt['mid'] != '').any() or (tgt['wikipedia_url'] != '').any()),
            first_mention_pct=tgt['first_mention_pct'].min(),
            names_used=' | '.join(sorted(set(' | '.join(tgt['mention_texts']).split(' | ')))),
            outranked_by_boilerplate=', '.join(above[above['boilerplate_hint']]['entity']),
        )
    res['in_title'] = contains_any(row['title'], names)
    res['in_h1'] = contains_any(row['h1'], names)
    res['in_subheadings'] = contains_any(row['subheadings'], names)
    res['primary_name_in_title'] = contains_any(row['title'], {norm(row['target_entity'])})

    r = res['target_rank']
    if r is None:
        score = 0
    elif r == 1 and res['kg_linked'] and (res['first_mention_pct'] or 0) <= 10:
        score = 3
    elif r <= 3:
        score = 2
    else:
        score = 1
    res['clarity_score_lite'] = score
    res['tips'] = ' • '.join(clarity_tips(res, row))
    return res

def clarity_tips(res, row):
    t, tips = row['target_entity'], []
    if res['target_rank'] is None:
        return [f'The API did not detect "{t}" at all. Name it explicitly, early and more than once.']
    if res['target_rank'] > 1:
        tips.append(f'"{res["top_entity"]}" is more salient than "{t}" (rank {res["target_rank"]}). Make {t} the grammatical subject of your opening sentences.')
    if res['outranked_by_boilerplate']:
        tips.append(f'Template or promo entities outrank the topic ({res["outranked_by_boilerplate"]}). Move CTAs, newsletter boxes and brand copy out of the article body or shorten them.')
    if (res['first_mention_pct'] or 0) > 10:
        tips.append(f'The first mention is {res["first_mention_pct"]:.0f}% into the text. Mention {t} in the first paragraph.')
    if not res['primary_name_in_title'] or not res['in_h1']:
        where = ' and '.join(x for x, ok in (('title', res['primary_name_in_title']), ('H1', res['in_h1'])) if not ok)
        tips.append(f'Use "{t}" in the {where}.')
    if not res['in_subheadings']:
        tips.append(f'Use {t} in at least one subheading.')
    if res['target_split_into'] > 1 and res['primary_share'] < 0.7:
        tips.append(f'The topic is split across {res["target_split_into"]} entity entries ({res["names_used"]}). Pick one primary name and introduce the others once, e.g. "{t} (also called ...)".')
    if not res['kg_linked']:
        tips.append(f'The API didn\'t link {t} to a Knowledge Graph entry. Use the most common, unambiguous name and add context that disambiguates it.')
    return tips or ['Clear: the target is the most salient entity. Keep it that way when you update the page.']

clarity = pd.DataFrame([clarity_row(i, ANALYSE_TEXT) for i in pages.index])
clarity[['url', 'role', 'top_entity', 'target_rank', 'best_alias_rank', 'target_salience', 'primary_share',
         'target_split_into', 'kg_linked', 'first_mention_pct', 'in_title', 'in_h1', 'clarity_score_lite']]

,url,role,top_entity,target_rank,best_alias_rank,target_salience,primary_share,target_split_into,kg_linked,first_mention_pct,in_title,in_h1,clarity_score_lite
0,https://www.brewly.example/blog/how-to-use-a-cafetiere/,yours,Brewly,13,2,0.1148,0.19,3,True,10.8,True,False,1
1,https://www.coffee-one.example/guides/how-to-use-a-french-press/,competitor,French press,1,1,0.3177,0.94,3,True,0.8,True,True,3
2,https://www.dailygrindnotes.example/brew/french-press-coffee/,competitor,coffee,2,2,0.1769,1.00,1,True,0.0,True,True,2


In [9]:
for _, r in clarity.iterrows():
    print(f"\n{r['role'].upper()}: {r['url']}  (lite clarity score {r['clarity_score_lite']}/3)")
    for tip in r['tips'].split(' • '):
        print('  -', tip)


YOURS: https://www.brewly.example/blog/how-to-use-a-cafetiere/  (lite clarity score 1/3)
  - "Brewly" is more salient than "French press" (rank 13). Make French press the grammatical subject of your opening sentences.
  - Template or promo entities outrank the topic (Brewly). Move CTAs, newsletter boxes and brand copy out of the article body or shorten them.
  - The first mention is 11% into the text. Mention French press in the first paragraph.
  - Use "French press" in the title and H1.
  - The topic is split across 3 entity entries (Cafetière | French press | cafetière | plunger pot | press pot). Pick one primary name and introduce the others once, e.g. "French press (also called ...)".

COMPETITOR: https://www.coffee-one.example/guides/how-to-use-a-french-press/  (lite clarity score 3/3)
  - Clear: the target is the most salient entity. Keep it that way when you update the page.

COMPETITOR: https://www.dailygrindnotes.example/brew/french-press-coffee/  (lite clarity score 2/3)
  

## 7. Your page vs competitors

The same target entity, side by side. A competitor page where the target is #1 with high salience shows what a clearly focused page on this topic looks like to the API. The gap is a rough measure of how much clearer your page could be.

In [10]:
comparison = clarity[['role', 'url', 'target_rank', 'target_salience', 'top_entity', 'top_salience', 'clarity_score_lite']].copy()
comp = comparison[comparison['role'] == 'competitor']
if len(comp):
    bench = comp['target_salience'].median()
    comparison['vs_competitor_median'] = (comparison['target_salience'] - bench).round(4)
    print(f'Competitor median salience for the target: {bench:.3f}')
comparison.sort_values(['role', 'target_salience'], ascending=[False, False])

Competitor median salience for the target: 0.247


,role,url,target_rank,target_salience,top_entity,top_salience,clarity_score_lite,vs_competitor_median
0,yours,https://www.brewly.example/blog/how-to-use-a-cafetiere/,13,0.1148,Brewly,0.1954,1,-0.1325
1,competitor,https://www.coffee-one.example/guides/how-to-use-a-french-press/,1,0.3177,French press,0.2993,3,0.0704
2,competitor,https://www.dailygrindnotes.example/brew/french-press-coffee/,2,0.1769,coffee,0.2376,2,-0.0704


## 8. Main text vs full page (your pages)

The same page analysed twice: once with only the article text and once with everything on the page, navigation and footer included. If the target drops further, or brand and template entities climb, in the full version, then anything that feeds a model the whole page (rather than a clean extraction) sees a less focused page. This is the effect Beatrice Gamba describes in *Why Navigation Elements Outrank Your Brand in Entity Analysis*.

In [11]:
if COMPARE_FULL_TEXT and ANALYSE_TEXT == 'main':
    full = pd.DataFrame([clarity_row(i, 'full') for i in pages.index[pages['role'] == 'yours']])
    both = pd.concat([clarity[clarity['role'] == 'yours'], full])
    display(both[['url', 'text_mode', 'top_entity', 'top_salience', 'target_rank', 'target_salience', 'outranked_by_boilerplate']])
    for url, g in entities[entities['role'] == 'yours'].groupby('url'):
        side = pd.concat({m: g[g['text_mode'] == m].head(8).reset_index(drop=True)[['entity', 'salience']]
                          for m in ['main', 'full']}, axis=1)
        print(f'\nTop 8 entities, {url}')
        display(side)
else:
    print('Set COMPARE_FULL_TEXT = True (with ANALYSE_TEXT = "main") to run this comparison.')

,url,text_mode,top_entity,top_salience,target_rank,target_salience,outranked_by_boilerplate
0,https://www.brewly.example/blog/how-to-use-a-cafetiere/,main,Brewly,0.1954,13,0.1148,Brewly
0,https://www.brewly.example/blog/how-to-use-a-cafetiere/,full,Brewly,0.3073,16,0.0988,"Brewly, newsletter"



Top 8 entities, https://www.brewly.example/blog/how-to-use-a-cafetiere/


main                   full         
           entity salience        entity salience
0          Brewly   0.1954        Brewly   0.3073
1       cafetière   0.0743     cafetière   0.0690
2          coffee   0.0731        coffee   0.0678
3    coffee beans   0.0549  coffee beans   0.0378
4           water   0.0510        kettle   0.0356
5           grind   0.0495         water   0.0356
6          filter   0.0480         grind   0.0348
7  Brewly Classic   0.0464        filter   0.0338

## 9. Export: Sheets-ready report and CSVs

`entity_salience_report.xlsx` (upload to Google Drive and open with Google Sheets) has:

* **How to read**: what salience is and isn't, and the score rubric.
* **Clarity**: one row per page, colour-coded score, target rank and tips.
* **Comparison**: your page vs competitors.
* **Entities**: every entity per page and text mode, with salience data bars and the target rows highlighted.

In [12]:
from openpyxl import Workbook
from openpyxl.formatting.rule import CellIsRule, ColorScaleRule, DataBarRule, FormulaRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

HEADER_FILL, HEADER_FONT = PatternFill('solid', fgColor='251046'), Font(bold=True, color='FFFFFF')

def write_table(ws, df, widths=None):
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append([None if (isinstance(v, float) and math.isnan(v)) else v for v in r])
    for c in ws[1]:
        c.fill, c.font, c.alignment = HEADER_FILL, HEADER_FONT, Alignment(wrap_text=True, vertical='top')
    ws.freeze_panes = 'B2'
    if ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions
    for i, name in enumerate(df.columns, start=1):
        w = (widths or {}).get(name) or min(max(len(str(name)), *(len(str(v)) for v in df[name].head(200)), 8) + 2, 50)
        ws.column_dimensions[get_column_letter(i)].width = w

def letter(df, name):
    return get_column_letter(list(df.columns).index(name) + 1)

wb = Workbook()
ws = wb.active
ws.title = 'How to read'
for r in [
    ['Entity salience checker', 'by Lazarina Stoy, MLforSEO. Lite Clarity check from The AI-powered Entity Audit (mlforseo.com/courses/the-ai-powered-entity-audit/)'],
    ['Data', 'SAMPLE: fictional pages, stand-in salience values in the v1 response format' if USE_SAMPLE else 'Google Cloud Natural Language API v1 analyzeEntities'],
    ['Salience', 'The Natural Language API\'s estimate (0-1) of how central an entity is to the text sent. Not a confirmed Google Search or LLM signal.'],
    ['Score 3', 'Target is #1, linked to the Knowledge Graph, first mentioned in the first 10% of the text'],
    ['Score 2', 'Target in the top 3 (or #1 missing one condition)'],
    ['Score 1', 'Target detected but outside the top 3'],
    ['Score 0', 'Target not detected'],
]:
    ws.append(r)
ws['A1'].font = Font(bold=True, size=14)
ws.column_dimensions['A'].width, ws.column_dimensions['B'].width = 16, 120

cl = clarity.drop(columns=['primary_name_in_title'])
ws_c = wb.create_sheet('Clarity')
write_table(ws_c, cl, widths={'url': 50, 'tips': 120, 'names_used': 30, 'outranked_by_boilerplate': 30})
n = ws_c.max_row
sc = letter(cl, 'clarity_score_lite')
for val, colour in [(0, 'F4C7C3'), (1, 'FCE8B2'), (2, 'C9DAF8'), (3, 'D9EAD3')]:
    ws_c.conditional_formatting.add(f'{sc}2:{sc}{n}', CellIsRule(operator='equal', formula=[str(val)], fill=PatternFill('solid', fgColor=colour)))
for row in ws_c.iter_rows(min_row=2):
    row[list(cl.columns).index('tips')].alignment = Alignment(wrap_text=True, vertical='top')

ws_p = wb.create_sheet('Comparison')
write_table(ws_p, comparison, widths={'url': 55})
ts = letter(comparison, 'target_salience')
ws_p.conditional_formatting.add(f'{ts}2:{ts}{ws_p.max_row}', DataBarRule(start_type='num', start_value=0, end_type='max', color='8448EF'))

ent_out = entities.drop(columns=[])
ws_e = wb.create_sheet('Entities')
write_table(ws_e, ent_out, widths={'url': 50, 'mention_texts': 40, 'wikipedia_url': 45})
m = ws_e.max_row
sl = letter(ent_out, 'salience')
ws_e.conditional_formatting.add(f'{sl}2:{sl}{m}', DataBarRule(start_type='num', start_value=0, end_type='max', color='8448EF'))
tg = letter(ent_out, 'is_target')
ws_e.conditional_formatting.add(f'A2:{get_column_letter(len(ent_out.columns))}{m}',
                                FormulaRule(formula=[f'${tg}2=TRUE'], fill=PatternFill('solid', fgColor='FFF2CC')))

os.makedirs('output', exist_ok=True)
wb.save('output/entity_salience_report.xlsx')
clarity.to_csv('output/clarity_by_page.csv', index=False)
entities.to_csv('output/entities.csv', index=False)
comparison.to_csv('output/comparison.csv', index=False)
print('Saved to output/:', sorted(os.listdir('output')))
try:
    from google.colab import files
    files.download('output/entity_salience_report.xlsx')
except ImportError:
    pass

Saved to output/: ['clarity_by_page.csv', 'comparison.csv', 'entities.csv', 'entity_salience_report.xlsx']


### Reading the sample result

In the sample, the fictional Brewly page is a cafetière guide. Its most salient entity is the brand, **Brewly**, followed by "cafetière". "French press" (the target, and the name linked to the Knowledge Graph) sits far down the list, because the page uses three names for one thing and only says "French press" at the very end. The aliases are found (`best_alias_rank`), but the topic's salience is split across separate, unlinked entity rows. The newsletter and shipping copy inside the article adds more brand and promo entities. Both competitor pages lead with "French press", use it in the title, H1 and subheadings, and score higher.

Remember that the sample salience values come from a stand-in formula, so the exact numbers mean nothing. The pattern is the point: run it with your key to get Google's real scores for your pages.

## Limits and honest caveats

* **Salience is one API's opinion.** It's the Natural Language API's estimate for the text you send. Google hasn't said Search or AI Overviews use it, and LLM-based assistants don't call this API. Use it to spot pages that are unclear to *a* machine reader, not as a ranking score.
* **The text you send changes the answer.** Main-text extraction can miss content (tabs, accordions, JavaScript-rendered text) or keep boilerplate. Check `main_text` for a page if a result looks odd. JavaScript-heavy pages may need a rendered crawl instead of `requests`.
* **Salience is relative within one document.** Comparing values across pages of very different lengths is rough. Ranks are more stable than raw numbers.
* **Knowledge Graph links aren't guaranteed.** Many valid entities come back without a MID, especially niche terms and brands.
* **Language support varies.** Entity analysis supports a limited set of languages; check the API's language support page for yours.
* **Costs.** Units are rounded up per request, so many short texts cost more than a few long ones.

## What to do next

* Fix the tips for your page, re-run, and compare the target's rank and salience before and after.
* Read Beatrice Gamba's [Why Navigation Elements Outrank Your Brand in Entity Analysis](https://www.mlforseo.com/machine-learning-tutorials/entity-analysis/why-navigation-elements-outrank-your-brand-in-entity-analysis/) for why template entities win.
* For entity extraction without code, use the free [Entity Analysis with Google Cloud Natural Language API (No-Code Template)](https://www.mlforseo.com/resource/entity-analysis-with-with-google-cloud-natural-language-api-no-code-template/); for a Python walkthrough of the API, see [Extracting Entities with Google Cloud Natural Language API in Python](https://www.mlforseo.com/resource/extracting-entities-with-google-cloud-natural-language-api-in-python-tutorial/).
* This is one check from the Clarity step. The full audit (Clarity, Coverage, Consistency, Connection and Corroboration, with a scorecard across a whole site) is in my course [The AI-powered Entity Audit](https://www.mlforseo.com/courses/the-ai-powered-entity-audit/).